# 1. Closed Queuing network simulation (17 pt)
Build a simulator to capture the following system and answer the questions below. A total of N=40 jobs circulate in different parts of the systems: CPU, disk, and resting area. There is one CPU and two disks, one slow and one fast. Each job starts at the CPU station and takes an average of 2 seconds. After CPU, a job fetches the data from the disk, needing an average of 3000 disk cycles. There are two possible choices of disks, fast and slow, which have a speed of 1000 cycles per second and 100 cycles per seconds, respectively. After the disk station, the job can rest for 15 seconds on average. The figure below gives an overview of the system:


![image](Queued.png)

The choices of distributions are up to you. You are welcomed to try different distributions to answer the following questions and assess the impact of different variants.


Questions:
(3p) What will be the maximal system throughput in terms of number of jobs, given two different load balancing strategies for sending jobs between the fast and slow disk? You need to come up with two load balancing strategies and compare them. 
(3p) What will the system throughput and average response time of a job be if a faster CPU is used? Say, CPU time is reduced to 1 seconds on average and the rest of the system remains the same.
(3p) What will the system throughput and average response time of a job be if a second fast disk is added? Again, you need to compare the throughput under two different load balancing strategies.
(3p) What will the system throughput and average response time of a job be if a faster CPU is used and a second fast disk is added? And, you use the better load balancing strategies out of two you propose in the first question?
(5pt) If you can answer all the above questions with different number of N and plot them in the following style:


![image](Graph.png)

In [1]:
# imports
import os
import sys
from contextlib import contextmanager, redirect_stdout
from typing import Generator

import matplotlib.pyplot as plt
import numpy as np
import simpy
from matplotlib.axes import Axes
from matplotlib.figure import Figure

RNG = np.random.default_rng(seed=0)

@contextmanager
def suppress_print():
    """Temporarily disable print statements."""
    with redirect_stdout(open(os.devnull, "w")):
        yield

In [ ]:
# 50/50 Closed Queueing Network Simulation
class ClosedQueueNetwork:
    def __init__(self, N_jobs=40):
        self.env = simpy.Environment()
        self.cpu = simpy.Resource(self.env, capacity=1)
        self.slow_disk = simpy.Resource(self.env, capacity=1)
        self.fast_disk = simpy.Resource(self.env, capacity=1)
        self.N_jobs = N_jobs

    def job_process(self, job_id):
        while True:
            # CPU phase
            with self.cpu.request() as req:
                yield req
                print(f"Time {self.env.now:.2f}: job {job_id} using CPU")
                yield self.env.timeout(np.random.exponential(2))  # mean 2s

            # Disk phase: randomly select fast or slow
            if np.random.rand() < 0.5:
                # Fast disk
                with self.fast_disk.request() as req:
                    yield req
                    disk_time = 3000 / 1000  # 3s on avg.
                    yield self.env.timeout(np.random.exponential(disk_time))
            else:
                # Slow disk
                with self.slow_disk.request() as req:
                    yield req
                    disk_time = 3000 / 100   # 30s on avg.
                    yield self.env.timeout(np.random.exponential(disk_time))
            
            # Resting
            print(f"Time {self.env.now:.2f}: job {job_id} resting")
            yield self.env.timeout(np.random.exponential(15))

    def run(self, until=1000):
        for i in range(self.N_jobs):
            self.env.process(self.job_process(i))
        self.env.run(until=until)


In [3]:
ClosedQueueNetwork(N_jobs=40).run(until=1000)

Time 0.00: job 0 using CPU
Time 0.44: job 1 using CPU
Time 2.46: job 2 using CPU
Time 8.57: job 3 using CPU
Time 10.65: job 4 using CPU
Time 10.71: job 0 resting
Time 11.69: job 1 resting
Time 13.09: job 2 resting
Time 15.29: job 5 using CPU
Time 16.12: job 6 using CPU
Time 16.24: job 7 using CPU
Time 16.51: job 5 resting
Time 16.78: job 8 using CPU
Time 18.30: job 6 resting
Time 21.86: job 9 using CPU
Time 22.08: job 10 using CPU
Time 22.41: job 7 resting
Time 23.33: job 11 using CPU
Time 24.67: job 8 resting
Time 25.26: job 10 resting
Time 25.68: job 12 using CPU
Time 27.39: job 13 using CPU
Time 27.84: job 14 using CPU
Time 28.28: job 11 resting
Time 30.36: job 12 resting
Time 30.48: job 15 using CPU
Time 32.93: job 16 using CPU
Time 34.03: job 17 using CPU
Time 34.12: job 18 using CPU
Time 34.84: job 19 using CPU
Time 35.83: job 20 using CPU
Time 36.57: job 21 using CPU
Time 37.50: job 22 using CPU
Time 38.79: job 13 resting
Time 38.99: job 23 using CPU
Time 39.05: job 24 using CPU

In [ ]:
# 50/50 Closed Queueing Network Simulation with Strategy Comparison
class ClosedQueueNetwork:
    def __init__(self, N_jobs=40, strategy="random"):
        self.env = simpy.Environment()
        self.cpu = simpy.Resource(self.env, capacity=1)
        self.slow_disk = simpy.Resource(self.env, capacity=1)
        self.fast_disk = simpy.Resource(self.env, capacity=1)
        self.N_jobs = N_jobs
        self.strategy = strategy
        self.completed_cycles = 0
        self.fast_prob = 1000 / (1000 + 100)
        
        # Utilization tracking
        self.cpu_busy_time = 0
        self.disk_fast_busy_time = 0
        self.disk_slow_busy_time = 0


    def job_process(self, job_id):
        while True:
            # CPU phase
            with self.cpu.request() as req:
                yield req
                cpu_service = np.random.exponential(2)
                self.cpu_busy_time += cpu_service
                yield self.env.timeout(cpu_service)

            # Disk choice based on strategy
            r = np.random.rand()
            
            if self.strategy == "random":
                use_fast = (r < 0.5)
            elif self.strategy == "proportional":
                use_fast = (r < self.fast_prob)
            elif self.strategy == "jsq":
                # Join-the-Shorter-Queue: pick disk with fewer waiting jobs
                fast_queue_len = len(self.fast_disk.queue)
                slow_queue_len = len(self.slow_disk.queue)
                if fast_queue_len < slow_queue_len:
                    use_fast = True
                elif slow_queue_len < fast_queue_len:
                    use_fast = False
                else:  # tied, prefer fast
                    use_fast = True
            else:
                raise ValueError(f"Unknown strategy: {self.strategy}")

            # Execute disk phase
            if use_fast:
                with self.fast_disk.request() as req:
                    yield req
                    disk_service = np.random.exponential(3)
                    self.disk_fast_busy_time += disk_service
                    yield self.env.timeout(disk_service)
            else:
                with self.slow_disk.request() as req:
                    yield req
                    disk_service = np.random.exponential(30)
                    self.disk_slow_busy_time += disk_service
                    yield self.env.timeout(disk_service)

            # Rest phase
            yield self.env.timeout(np.random.exponential(15))
            self.completed_cycles += 1


    def run(self, until=100000):
        for i in range(self.N_jobs):
            self.env.process(self.job_process(i))
        self.env.run(until=until)


    def system_throughput(self):
        """Return completed cycles per unit time."""
        return self.completed_cycles / self.env.now
    
    
    def utilization(self):
        """Return resource utilization (fraction of time busy)."""
        total_time = self.env.now
        return {
            "cpu": self.cpu_busy_time / total_time,
            "disk_fast": self.disk_fast_busy_time / total_time,
            "disk_slow": self.disk_slow_busy_time / total_time
        }


def compare_strategies(sim_lengths=[10000, 50000, 100000], n_reps=5):
    strategies = ["random", "proportional", "jsq"]
    results = {}
    utilizations = {}
    
    for strategy in strategies:
        throughputs = np.empty((len(sim_lengths), n_reps))
        utils = []
        
        for i, length in enumerate(sim_lengths):
            for j in range(n_reps):
                net = ClosedQueueNetwork(N_jobs=40, strategy=strategy)
                net.run(length)
                throughputs[i][j] = net.system_throughput()
                if i == len(sim_lengths) - 1:  # only for longest sim
                    utils.append(net.utilization())
        
        results[strategy] = throughputs
        utilizations[strategy] = utils
    
    # Print detailed summary
    print("\n" + "="*80)
    print("SYSTEM THROUGHPUT COMPARISON")
    print("="*80)
    
    for strategy in strategies:
        print(f"\n{strategy.upper()} STRATEGY:")
        mean_thr = results[strategy].mean(axis=-1)
        std_thr = results[strategy].std(axis=-1)
        min_thr = results[strategy].min(axis=-1)
        max_thr = results[strategy].max(axis=-1)
        
        for i, length in enumerate(sim_lengths):
            print(f"  Length {length:6d}: mean={mean_thr[i]:.4f}, std={std_thr[i]:.4f}, "
                  f"min={min_thr[i]:.4f}, max={max_thr[i]:.4f}")
    
    # Print utilization for longest simulation
    print(f"\n" + "="*80)
    print("RESOURCE UTILIZATION (at sim length {})".format(sim_lengths[-1]))
    print("="*80)
    
    for strategy in strategies:
        utils_at_longest = utilizations[strategy]
        avg_cpu = np.mean([u["cpu"] for u in utils_at_longest])
        avg_fast = np.mean([u["disk_fast"] for u in utils_at_longest])
        avg_slow = np.mean([u["disk_slow"] for u in utils_at_longest])
        
        print(f"\n{strategy.upper()}:")
        print(f"  CPU utilization:       {avg_cpu:.2%}")
        print(f"  Fast disk utilization: {avg_fast:.2%}")
        print(f"  Slow disk utilization: {avg_slow:.2%}")
    
    # Print comparisons
    print(f"\n" + "="*80) # for some pretty lines ^^
    print("COMPARISON (vs RANDOM, at longest sim)")
    print("="*80)
    
    random_thr = results["random"][-1].mean()
    
    for strategy in ["proportional", "jsq"]:
        strat_thr = results[strategy][-1].mean()
        ratio = strat_thr / random_thr
        improvement = (ratio - 1) * 100
        print(f"{strategy.upper():15s}: {ratio:.2f}x throughput, {improvement:+.1f}% improvement")
    
    return results, utilizations


# Run the comparison
results, utilizations = compare_strategies()



SYSTEM THROUGHPUT COMPARISON

RANDOM STRATEGY:
  Length  10000: mean=0.0749, std=0.0068, min=0.0681, max=0.0875
  Length  50000: mean=0.0653, std=0.0004, min=0.0647, max=0.0659
  Length 100000: mean=0.0680, std=0.0013, min=0.0659, max=0.0693

PROPORTIONAL STRATEGY:
  Length  10000: mean=0.3582, std=0.0042, min=0.3526, max=0.3637
  Length  50000: mean=0.3591, std=0.0035, min=0.3530, max=0.3637
  Length 100000: mean=0.3573, std=0.0024, min=0.3532, max=0.3608

JSQ STRATEGY:
  Length  10000: mean=0.3633, std=0.0066, min=0.3549, max=0.3717
  Length  50000: mean=0.3669, std=0.0022, min=0.3643, max=0.3708
  Length 100000: mean=0.3660, std=0.0016, min=0.3640, max=0.3688

RESOURCE UTILIZATION (at sim length 100000)

RANDOM:
  CPU utilization:       13.80%
  Fast disk utilization: 10.25%
  Slow disk utilization: 100.01%

PROPORTIONAL:
  CPU utilization:       71.61%
  Fast disk utilization: 97.40%
  Slow disk utilization: 96.78%

JSQ:
  CPU utilization:       73.21%
  Fast disk utilization: 99.